Make sure the right schema is used

In [0]:
USE CATALOG sac;
USE SCHEMA customer_service;

In [0]:
select zone, count(*) from customer group by zone

zone,count(1)
Thüringen,31
Schleswig-Holstein,9471
Hamburg,3343
Sachsen,11507
Hessen,26
Berlin,4766
Nordrhein-Westfalen,51939
Brandenburg,10013
Niedersachsen,1
null,8903


# Gold Tables
average customer

In [0]:
CREATE OR REPLACE VIEW average_customer AS
SELECT
    zone,
    COUNT(*) AS amount_customers,
    ROUND(AVG(monthly_bill), 2) AS avg_monthly_bill,
    ROUND(AVG(speed_tier_mbps), 0) AS avg_speed_tier,
    ROUND(AVG(data_usage_gb_last_month), 2) AS avg_data_usage
FROM
    customer
GROUP BY
    zone;

tickets per customer

In [0]:
CREATE OR REPLACE VIEW customer_connection_ticket_count AS
SELECT
    c.customer_id,
    COUNT(DISTINCT l.timestamp) AS connection_fails,
    COUNT(DISTINCT s.ticket_id) AS tickets,
    COUNT(DISTINCT ca.session_id) AS chats,
    ch.churned as churned
FROM
    customer c
        JOIN ticket s
            ON c.customer_id = s.customer_id
        LEFT JOIN log l
            ON c.customer_id = l.customer_id
            AND l.issue_detected != 'none'
        LEFT JOIN churn ch
            ON c.customer_id = ch.customer_id
        LEFT JOIN chat ca
            ON c.customer_id = ca.customer_id
GROUP BY
    c.customer_id, ch.churned;

churned customer

In [0]:
CREATE OR REPLACE VIEW churned_customer_details AS
SELECT
    c.customer_id,
    c.speed_tier_mbps,
    c.monthly_bill,
    COUNT(DISTINCT l.timestamp) AS connection_fails,
    COUNT(DISTINCT s.ticket_id) AS tickets,
    COUNT(DISTINCT ca.session_id) AS chats
FROM
    customer c
        LEFT JOIN ticket s
            ON c.customer_id = s.customer_id
        LEFT JOIN log l
            ON c.customer_id = l.customer_id
            AND l.issue_detected != 'none'
        LEFT JOIN chat ca
            ON c.customer_id = ca.customer_id
        JOIN churn ch
            ON c.customer_id = ch.customer_id
WHERE
    ch.churned = true
GROUP BY
    c.customer_id,
    c.speed_tier_mbps,
    c.monthly_bill;

location detail

In [0]:
CREATE OR REPLACE VIEW location_detail AS
WITH revenue_per_location AS (
    SELECT
        zone,
        SUM(monthly_bill) AS revenue
    FROM
        customer
    GROUP BY
        zone
),
issues_per_location AS (
    SELECT
        c.zone,
        COUNT(
            CASE
                WHEN l.issue_detected != 'none' THEN 1
            END
        ) AS issue_count
    FROM
        customer c
            LEFT JOIN log l
                ON c.customer_id = l.customer_id
    GROUP BY
        c.zone
)
SELECT
    c.zone,
    COUNT(DISTINCT c.customer_id) AS customer_count,
    ROUND(r.revenue / 1000, 2) AS revenue_in_t,
    i.issue_count AS issue_count,
    COUNT(DISTINCT t.ticket_id) AS ticket_count,
    COUNT(DISTINCT ch.session_id) AS chat_count
FROM
    customer c
        LEFT JOIN revenue_per_location r
            ON c.zone = r.zone
        LEFT JOIN issues_per_location i
            ON c.zone = i.zone
        LEFT JOIN ticket t
            ON c.customer_id = t.customer_id
        LEFT JOIN chat ch
            ON c.customer_id = ch.customer_id
GROUP BY
    c.zone,
    r.revenue,
    i.issue_count;

average connection quality

In [0]:
CREATE OR REPLACE VIEW average_connection_quality AS
SELECT
    c.zone,
    ROUND(AVG(l.speed_measured_mbps), 0) AS avg_speed,
    ROUND(AVG(l.packet_loss_percent), 2) AS avg_packet_loss,
    ROUND(AVG(l.latency_ms), 2) AS avg_latency,
    ROUND(AVG(l.downtime_minutes), 2) AS avg_downtime,
    ROUND(AVG(l.connection_drops_count), 2) AS avg_connection_drops,
    COUNT(
        CASE
            WHEN l.issue_detected != 'none' THEN 1
            ELSE 0
        END
    ) AS count_issues
FROM
    log l
        LEFT JOIN customer c
            ON l.customer_id = c.customer_id
GROUP BY
    zone;

chat issues

In [0]:
CREATE OR REPLACE VIEW chat_issues AS
SELECT
    c.classification,
    m.sentiment,
    COUNT(m.sentiment) AS count,
    FIRST(c.comment) AS exmp_comment
FROM
    chat c
    LEFT JOIN message m
WHERE
    classification IS NOT NULL
    AND m.speaker = 'customer'
GROUP BY
    c.classification,
    m.sentiment
ORDER BY
    count DESC

sentiment for agent

In [0]:
CREATE OR REPLACE VIEW sentiment_for_agent AS
SELECT
    CONCAT(a.first_name, ' ', a.last_name) AS agent_name,
    m.sentiment,
    COUNT(m.sentiment) AS count
FROM
    chat c
        JOIN message m
            ON c.session_id = m.session_id
            AND m.speaker = 'customer'
        LEFT JOIN agent a
            ON c.agent_id = a.agent_id
GROUP BY
    agent_name,
    m.sentiment;

# Show tables

In [0]:
SELECT * FROM average_customer;

zone,amount_customers,avg_monthly_bill,avg_speed_tier,avg_data_usage
Thüringen,31,386.82,365.0,null
Schleswig-Holstein,9471,296.07,306.0,null
Hamburg,3343,291.09,311.0,null
Sachsen,11507,298.32,304.0,null
Hessen,26,260.93,352.0,null
Berlin,4766,298.77,314.0,null
Nordrhein-Westfalen,51939,302.09,306.0,null
Brandenburg,10013,299.89,305.0,null
Niedersachsen,1,838.1,1000.0,null
null,8903,296.82,307.0,null


In [0]:
SELECT * FROM customer_connection_ticket_count ORDER BY tickets DESC LIMIT 20;

customer_id,connection_fails,tickets,chats,churned
CUST_00739,19,3,2,false
CUST_01086,8,3,0,false
CUST_00865,21,3,2,false
CUST_01021,9,3,0,false
CUST_01051,9,3,0,false
CUST_00971,18,3,0,true
CUST_01130,12,2,0,false
CUST_00557,14,2,0,false
CUST_00712,10,2,2,false
CUST_00106,26,2,1,true


In [0]:
SELECT * FROM churned_customer_details ORDER BY tickets DESC LIMIT 20;

customer_id,speed_tier_mbps,monthly_bill,connection_fails,tickets,chats
CUST_00971,200,238.3,18,3,0
CUST_00695,50,223.4,15,2,0
CUST_00937,50,22.9,15,2,2
CUST_00615,1000,11.6,22,2,0
CUST_00284,50,207.4,18,2,0
CUST_00784,50,231.8,16,2,2
CUST_00021,200,677.0,18,2,0
CUST_00577,50,132.5,12,2,2
CUST_00106,50,216.3,26,2,1
CUST_00508,1000,71.2,13,2,1


In [0]:
SELECT * FROM location_detail ORDER BY customer_count DESC;

zone,customer_count,revenue_in_t,issue_count,ticket_count,chat_count
Nordrhein-Westfalen,51939,15690.14,627034,239,249
Sachsen,11507,3432.82,127574,48,56
Brandenburg,10013,3002.76,109976,57,58
Schleswig-Holstein,9471,2804.05,107475,52,47
null,8903,null,null,55,46
Berlin,4766,1423.94,53005,29,19
Hamburg,3343,973.1,38492,20,20
Thüringen,31,11.99,268,0,0
Hessen,26,6.78,376,0,0
Niedersachsen,1,0.84,6,0,0


In [0]:
SELECT * FROM average_connection_quality;

zone,avg_speed,avg_packet_loss,avg_latency,avg_downtime,avg_connection_drops,count_issues
Niedersachsen,871.0,2.01,43.02,0.0,0.38,34
Thüringen,314.0,3.55,45.49,3.5,2.73,1559
Schleswig-Holstein,363.0,3.05,44.15,2.75,2.2,695410
Sachsen,256.0,3.31,46.22,3.03,2.47,742865
Hamburg,262.0,3.33,47.53,3.09,2.44,211095
Hessen,264.0,3.02,52.84,2.61,2.13,1856
null,264.0,3.41,47.29,3.21,2.57,554339
Berlin,320.0,2.83,40.99,2.2,1.75,429487
Nordrhein-Westfalen,260.0,2.82,42.63,2.35,1.96,4373402
Brandenburg,243.0,2.98,43.92,2.73,2.18,725222


In [0]:
SELECT * FROM chat_issues order by classification, sentiment;

classification,sentiment,count,exmp_comment
OTHER,Negativ,26000,Freundliche Verabschiedung
OTHER,Neutral,46700,Überweisung erfolgt innerhalb von 3-5 Werktagen.
OTHER,Positiv,5700,Überweisung erfolgt innerhalb von 3-5 Werktagen.
OTHER,Unbekannt,1250,Datumsproblem stört die Validierung der SSL-Zertifikate
PRICE,Negativ,19760,Rechnungen belaufen sich auf 40 Euro monatlich
PRICE,Neutral,35492,Rechnungen belaufen sich auf 40 Euro monatlich
PRICE,Positiv,4332,Rechnungen belaufen sich auf 40 Euro monatlich
PRICE,Unbekannt,950,Rechnungen belaufen sich auf 40 Euro monatlich
SERVICE,Negativ,211120,Technikertermin wird vereinbart
SERVICE,Neutral,379204,Drittanbieter- und Sonderrufnummernsperre ist eingerichtet


In [0]:
SELECT * FROM sentiment_for_agent ORDER BY agent_name, sentiment LIMIT 20;

agent_name,sentiment,count
Anna Schmidt,Negativ,27
Anna Schmidt,Neutral,45
Anna Schmidt,Positiv,4
Anna Schmidt,Unbekannt,2
Ben Neumann,Negativ,24
Ben Neumann,Neutral,43
Ben Neumann,Positiv,3
David Wolf,Negativ,25
David Wolf,Neutral,41
David Wolf,Positiv,6
